# 01 — Population Mortality EDA: France (HMD)

## Objective

This notebook performs an **exploratory data analysis** of French population mortality data
sourced from the Human Mortality Database (HMD). The analysis covers ages 40 to 100 and
calendar years 1968 to 2022.

We examine:

- **Lexis diagrams** — heatmaps of log-mortality rates over the age-year surface
- **Mortality trends by age** — how rates evolve over time for key ages
- **Mortality curves by period** — age-specific profiles at selected calendar years
- **Life expectancy trends** — period life expectancy at age 40 over time
- **Summary statistics** — descriptive statistics of the mortality surface
- **Interactive mortality surface** — Plotly heatmap for deeper exploration
- **COVID-19 impact** — excess mortality visible in 2020

If HMD data files are not found locally, the notebook generates **synthetic mortality data**
using a Gompertz model with historical improvement rates, providing a realistic fallback
for development and demonstration.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go

# Allow imports from the project src/ directory
sys.path.insert(0, str(Path("..").resolve()))

from src.mortality_tables import build_life_table

# Global plot settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

print("Imports OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1. Load or Generate Mortality Data

We attempt to read HMD death-rate files from `../data/raw/`. If the files are not present,
we generate a synthetic mortality surface using the **Gompertz law**:

$$\mu(x) = A \cdot e^{B \cdot x}$$

with a log-linear annual improvement of 1.5 % applied relative to a base year, and an
additional COVID-19 shock in 2020.

In [ ]:
AGES = np.arange(40, 101)       # 40 to 100 inclusive
YEARS = np.arange(1968, 2023)   # 1968 to 2022 inclusive
BASE_YEAR = 2000

# Gompertz parameters (realistic for French population)
A = 0.0001
B = 0.1
IMPROVEMENT_RATE = 0.015  # 1.5% annual mortality improvement


def generate_synthetic_mortality(ages, years, A, B, improvement_rate, base_year=2000):
    """
    Generate a synthetic mortality surface using the Gompertz model
    with log-linear mortality improvements and a COVID-19 shock in 2020.

    Returns a DataFrame of qx values with index=ages, columns=years.
    """
    rng = np.random.default_rng(42)
    qx_matrix = np.zeros((len(ages), len(years)))

    for j, year in enumerate(years):
        # Gompertz force of mortality at base year
        mu_base = A * np.exp(B * ages)

        # Apply improvement: mu(x,t) = mu(x, base) * exp(-improvement * (t - base))
        # Improvement is slightly stronger at younger ages
        age_factor = 1.0 - 0.005 * (ages - 40) / 60  # tapers from 1.0 to 0.95
        improvement = improvement_rate * age_factor
        mu_xt = mu_base * np.exp(-improvement * (year - base_year))

        # COVID-19 shock in 2020: +15% excess mortality, concentrated at ages 60+
        if year == 2020:
            covid_shock = 1.0 + 0.15 * np.clip((ages - 50) / 30, 0, 1)
            mu_xt = mu_xt * covid_shock

        # Convert force of mortality to qx: qx = 1 - exp(-mu)
        qx = 1 - np.exp(-mu_xt)

        # Add small random noise (Poisson-like variance)
        noise = rng.normal(0, 0.02, len(ages)) * qx
        qx = np.clip(qx + noise, 1e-6, 1.0)

        qx_matrix[:, j] = qx

    return pd.DataFrame(qx_matrix, index=ages, columns=years)


# Try loading HMD data
hmd_path = Path("..") / "data" / "raw"
hmd_files = list(hmd_path.glob("*.txt")) + list(hmd_path.glob("*.csv"))

if hmd_files:
    print(f"Found {len(hmd_files)} HMD file(s) — attempting to load...")
    # Standard HMD Mx format: Year, Age, Female, Male, Total
    try:
        df_raw = pd.read_csv(hmd_files[0], sep=r"\s+", skiprows=2, na_values=".")
        df_raw.columns = [c.strip() for c in df_raw.columns]
        # Pivot to matrix form
        df_raw["Age"] = df_raw["Age"].replace("110+", "110").astype(int)
        qx_surface = df_raw.pivot(index="Age", columns="Year", values="Total")
        qx_surface = qx_surface.loc[AGES[0]:AGES[-1], YEARS[0]:YEARS[-1]]
        data_source = "HMD"
        print("HMD data loaded successfully.")
    except Exception as e:
        print(f"Failed to parse HMD file: {e}")
        print("Falling back to synthetic data.")
        qx_surface = generate_synthetic_mortality(AGES, YEARS, A, B, IMPROVEMENT_RATE, BASE_YEAR)
        data_source = "Synthetic (Gompertz)"
else:
    print("No HMD files found in ../data/raw/ — generating synthetic mortality data.")
    qx_surface = generate_synthetic_mortality(AGES, YEARS, A, B, IMPROVEMENT_RATE, BASE_YEAR)
    data_source = "Synthetic (Gompertz)"

print(f"\nData source: {data_source}")
print(f"Shape: {qx_surface.shape[0]} ages x {qx_surface.shape[1]} years")
print(f"Ages: {qx_surface.index.min()} to {qx_surface.index.max()}")
print(f"Years: {qx_surface.columns.min()} to {qx_surface.columns.max()}")

## 2. Lexis Diagram — Log-Mortality Heatmap

The Lexis diagram visualises the entire mortality surface on a (year, age) grid.
We plot **log(qx)** so that the exponential age pattern is compressed and temporal
trends become visible. Diagonal ridges correspond to birth cohort effects.

In [ ]:
log_qx = np.log(qx_surface)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.pcolormesh(
    qx_surface.columns.values,
    qx_surface.index.values,
    log_qx.values,
    cmap="YlOrRd",
    shading="nearest",
)
cbar = fig.colorbar(im, ax=ax, label="ln(qx)")
ax.set_xlabel("Year")
ax.set_ylabel("Age")
ax.set_title(f"Lexis Diagram — Log-Mortality Surface, France ({data_source})")
ax.set_xlim(YEARS[0], YEARS[-1])
ax.set_ylim(AGES[0], AGES[-1])

# Annotate COVID-19 column
ax.axvline(x=2020, color="white", linewidth=1.0, linestyle="--", alpha=0.8)
ax.text(2020.5, 42, "COVID-19", color="white", fontsize=8, fontweight="bold", rotation=90)

plt.tight_layout()
plt.show()

## 3. Mortality Rates Over Time — Selected Ages

We trace the evolution of qx at five representative ages (50, 60, 70, 80, 90)
to visualise the pace and age-dependence of mortality improvements.

In [ ]:
selected_ages = [50, 60, 70, 80, 90]
palette = sns.color_palette("viridis", len(selected_ages))

fig, ax = plt.subplots(figsize=(12, 6))

for i, age in enumerate(selected_ages):
    ax.semilogy(
        qx_surface.columns,
        qx_surface.loc[age],
        label=f"Age {age}",
        color=palette[i],
        linewidth=1.5,
    )

ax.set_xlabel("Year")
ax.set_ylabel("qx (log scale)")
ax.set_title("Mortality Rates Over Time — Selected Ages")
ax.legend(title="Age", loc="upper right")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)

plt.tight_layout()
plt.show()

## 4. Mortality Rates by Age — Selected Calendar Years

A cross-sectional view: the age profile of mortality for four landmark periods.
The shift downward (and rightward on a log scale) reflects decades of improvement.

In [ ]:
selected_years = [1970, 1990, 2010, 2022]
# Ensure all selected years exist in the data
selected_years = [y for y in selected_years if y in qx_surface.columns]

palette_yr = sns.color_palette("coolwarm", len(selected_years))

fig, ax = plt.subplots(figsize=(12, 6))

for i, year in enumerate(selected_years):
    ax.semilogy(
        qx_surface.index,
        qx_surface[year],
        label=str(year),
        color=palette_yr[i],
        linewidth=1.8,
    )

ax.set_xlabel("Age")
ax.set_ylabel("qx (log scale)")
ax.set_title("Mortality Rates by Age — Selected Calendar Years")
ax.legend(title="Year", loc="upper left")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)

plt.tight_layout()
plt.show()

## 5. Life Expectancy Trends

We compute the **period life expectancy at age 40** for every calendar year using
the `build_life_table` function from `src/mortality_tables.py`.

In [ ]:
e40_series = {}

for year in qx_surface.columns:
    qx_col = qx_surface[year].values
    lt = build_life_table(qx_col, age_start=40)
    e40_series[year] = lt.loc[lt["age"] == 40, "ex"].iloc[0]

e40 = pd.Series(e40_series, name="e40")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(e40.index, e40.values, color="steelblue", linewidth=2)
ax.fill_between(e40.index, e40.values, alpha=0.15, color="steelblue")

# Highlight COVID year
if 2020 in e40.index:
    ax.scatter([2020], [e40[2020]], color="red", s=80, zorder=5, label="2020 (COVID-19)")
    ax.annotate(
        f"  {e40[2020]:.1f} yrs",
        xy=(2020, e40[2020]),
        fontsize=9,
        color="red",
    )

ax.set_xlabel("Year")
ax.set_ylabel("Period Life Expectancy at Age 40 (years)")
ax.set_title("Life Expectancy at Age 40 Over Time — France")
ax.legend()

plt.tight_layout()
plt.show()

print(f"e40 in {YEARS[0]}: {e40.iloc[0]:.2f} years")
print(f"e40 in {YEARS[-1]}: {e40.iloc[-1]:.2f} years")
print(f"Total gain:  {e40.iloc[-1] - e40.iloc[0]:.2f} years over {YEARS[-1] - YEARS[0]} years")

## 6. Summary Statistics

Descriptive statistics of the mortality surface and annual average qx.

In [ ]:
print("=" * 60)
print("MORTALITY SURFACE — SUMMARY STATISTICS")
print("=" * 60)

summary = qx_surface.stack().describe()
summary.index.name = "statistic"
summary.name = "qx"
print(summary.to_string())

print("\n" + "-" * 60)
print("Mean qx by Decade and Age Group")
print("-" * 60)

# Create age groups
age_bins = [40, 50, 60, 70, 80, 90, 101]
age_labels = ["40-49", "50-59", "60-69", "70-79", "80-89", "90-100"]

# Reshape to long format for grouping
qx_long = qx_surface.stack().reset_index()
qx_long.columns = ["age", "year", "qx"]
qx_long["age_group"] = pd.cut(qx_long["age"], bins=age_bins, labels=age_labels, right=False)
qx_long["decade"] = (qx_long["year"] // 10) * 10

pivot_summary = qx_long.groupby(["age_group", "decade"])["qx"].mean().unstack("decade")
print(pivot_summary.round(5).to_string())

In [ ]:
# Boxplot of qx distributions by decade
fig, ax = plt.subplots(figsize=(12, 5))

qx_long_sample = qx_long.copy()
qx_long_sample["log_qx"] = np.log10(qx_long_sample["qx"])

sns.boxplot(
    data=qx_long_sample,
    x="decade",
    y="log_qx",
    palette="Blues",
    ax=ax,
    fliersize=1.5,
)
ax.set_xlabel("Decade")
ax.set_ylabel("log10(qx)")
ax.set_title("Distribution of Mortality Rates by Decade (All Ages)")

plt.tight_layout()
plt.show()

## 7. Interactive Mortality Surface — Plotly

An interactive heatmap allows zooming into specific regions of the mortality surface.
Hover to read exact qx values at any (age, year) coordinate.

In [ ]:
fig_plotly = go.Figure(
    data=go.Heatmap(
        z=np.log(qx_surface.values),
        x=qx_surface.columns.tolist(),
        y=qx_surface.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="ln(qx)"),
        hovertemplate=(
            "Year: %{x}<br>"
            "Age: %{y}<br>"
            "ln(qx): %{z:.3f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_plotly.update_layout(
    title=f"Interactive Mortality Surface — France ({data_source})",
    xaxis_title="Year",
    yaxis_title="Age",
    width=900,
    height=600,
    template="plotly_white",
)

# Add COVID annotation
fig_plotly.add_vline(x=2020, line_dash="dash", line_color="black", opacity=0.5)
fig_plotly.add_annotation(
    x=2020, y=AGES[-1] + 1,
    text="COVID-19",
    showarrow=False,
    font=dict(size=10, color="red"),
)

fig_plotly.show()

## 8. COVID-19 Impact Analysis (2020)

We quantify the **excess mortality** in 2020 relative to the pre-pandemic trend.
The baseline is the average of 2017-2019 rates, and we measure how much 2020
deviated from this expected level at each age.

In [ ]:
# Baseline: average of 2017-2019
baseline_years = [y for y in [2017, 2018, 2019] if y in qx_surface.columns]
if baseline_years and 2020 in qx_surface.columns:
    qx_baseline = qx_surface[baseline_years].mean(axis=1)
    qx_2020 = qx_surface[2020]

    excess_ratio = (qx_2020 - qx_baseline) / qx_baseline * 100  # percent excess

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left panel: qx comparison
    ax1 = axes[0]
    ax1.semilogy(AGES, qx_baseline, label="Baseline (2017-2019 avg)", color="steelblue", linewidth=2)
    ax1.semilogy(AGES, qx_2020, label="2020 (observed)", color="crimson", linewidth=2)
    ax1.fill_between(
        AGES,
        qx_baseline,
        qx_2020,
        where=qx_2020 > qx_baseline,
        alpha=0.25,
        color="red",
        label="Excess mortality",
    )
    ax1.set_xlabel("Age")
    ax1.set_ylabel("qx (log scale)")
    ax1.set_title("2020 vs Pre-Pandemic Baseline")
    ax1.legend(fontsize=9)

    # Right panel: percentage excess
    ax2 = axes[1]
    colors = ["crimson" if v > 0 else "steelblue" for v in excess_ratio]
    ax2.bar(AGES, excess_ratio, color=colors, width=1.0, alpha=0.8)
    ax2.axhline(y=0, color="black", linewidth=0.8)
    ax2.set_xlabel("Age")
    ax2.set_ylabel("Excess Mortality (%)")
    ax2.set_title("Excess Mortality in 2020 vs 2017-2019 Baseline")

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f"Mean excess mortality (ages 40-100): {excess_ratio.mean():.1f}%")
    print(f"Peak excess mortality: {excess_ratio.max():.1f}% at age {excess_ratio.idxmax()}")
    print(f"Ages 60+: {excess_ratio.loc[60:].mean():.1f}% average excess")
    print(f"Ages 80+: {excess_ratio.loc[80:].mean():.1f}% average excess")
else:
    print("Required years (2017-2020) not available for COVID-19 analysis.")

In [ ]:
# Life expectancy drop in 2020
if 2019 in e40.index and 2020 in e40.index:
    drop = e40[2019] - e40[2020]
    print(f"Life expectancy at 40 in 2019: {e40[2019]:.2f} years")
    print(f"Life expectancy at 40 in 2020: {e40[2020]:.2f} years")
    print(f"Drop due to COVID-19:          {drop:.2f} years")
    if 2021 in e40.index:
        recovery = e40[2021] - e40[2020]
        print(f"Recovery in 2021:              +{recovery:.2f} years")

## Key Takeaways

1. **Steady improvement** — French mortality rates have fallen significantly across all ages
   since the late 1960s, with life expectancy at 40 gaining several years.

2. **Age gradient** — Improvements are relatively larger at younger ages (50-60) than at
   very old ages (90+), consistent with the rectangularisation of the survival curve.

3. **COVID-19** — The 2020 pandemic caused a visible spike in mortality, especially at ages
   60 and above, temporarily reversing years of progress.

4. **Lexis structure** — The heatmap reveals both period effects (vertical bands, e.g. 2020)
   and the overall downward drift in mortality over time.

These observations motivate the next notebook, where we fit a **Lee-Carter model** to
decompose the mortality surface into age, period, and trend components.